# Orquestador configurable de producción - rues-linker 0.22.4

Este notebook es una capa **delgada y declarativa**. La celda de parámetros define rutas, contratos `SourceSpec`, perfil y presupuestos; la librería realiza preflight, ingestión, smoke test, enlace, invariantes, publicación y metadatos.

El flujo admite **N fuentes** agregando elementos a `FUENTES`. DuckDB proyecta, normaliza, limita y colapsa en disco; solo los representantes exactos entran al matcher. La correlativa se expande y el payload se vuelve a adjuntar en Parquet sin materializar la salida completa. Esto reduce fuertemente la RAM, pero no promete memoria total cero: el matcher aún procesa en memoria el conjunto compacto de representantes.

Ejecución recomendada: primero deje `CORRIDA_COMPLETA = False`; revise el QA acotado y los metadatos; solo entonces active el universo completo.

## 1. Entorno reproducible

En local, abra el notebook desde el checkout 0.22.4. En Colab, suba el wheel con el nombre exacto `rues_linker-0.22.4-py3-none-any.whl` o defina `RUES_LINKER_WHEEL` con su ruta exacta. También puede definir `RUES_LINKER_PROJECT_DIR` para un checkout exacto. No se buscan ni eligen wheels por glob, fecha o nombre aproximado.

In [ ]:
from __future__ import annotations

import importlib
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

VERSION_OBJETIVO = "0.22.4"
_hay_google = importlib.util.find_spec("google") is not None  # si falta, find_spec("google.colab") revienta
ES_COLAB = _hay_google and importlib.util.find_spec("google.colab") is not None
_RUTA_WHEEL_CONFIG = os.environ.get("RUES_LINKER_WHEEL")
_RUTA_PROYECTO_CONFIG = os.environ.get("RUES_LINKER_PROJECT_DIR")
if _RUTA_WHEEL_CONFIG and _RUTA_PROYECTO_CONFIG:
    raise ValueError("Defina solo RUES_LINKER_WHEEL o RUES_LINKER_PROJECT_DIR.")

_objetivo_instalacion: Path | None = None
if _RUTA_WHEEL_CONFIG:
    _wheel = Path(_RUTA_WHEEL_CONFIG).expanduser().resolve()
    if not _wheel.is_file() or _wheel.suffix.casefold() != ".whl":
        raise FileNotFoundError(f"Wheel explícito inválido: {_wheel}")
    _objetivo_instalacion = _wheel
elif _RUTA_PROYECTO_CONFIG:
    _proyecto = Path(_RUTA_PROYECTO_CONFIG).expanduser().resolve()
    if (
        not (_proyecto / "pyproject.toml").is_file()
        or not (_proyecto / "src" / "record_linkage" / "__init__.py").is_file()
    ):
        raise FileNotFoundError(f"Checkout explícito inválido: {_proyecto}")
    _objetivo_instalacion = _proyecto
else:
    _cwd = Path.cwd().resolve()
    _checkout_local = _cwd.parent if _cwd.name == "notebooks" else _cwd
    _wheel_colab_exacto = Path("/content/rues_linker-0.22.4-py3-none-any.whl")
    if ES_COLAB and _wheel_colab_exacto.is_file():
        _objetivo_instalacion = _wheel_colab_exacto
    elif (_checkout_local / "pyproject.toml").is_file() and (
        _checkout_local / "src" / "record_linkage" / "__init__.py"
    ).is_file():
        _objetivo_instalacion = _checkout_local
    elif importlib.util.find_spec("record_linkage") is None:
        raise RuntimeError(
            "No hay una instalación verificable. Configure RUES_LINKER_WHEEL "
            "o RUES_LINKER_PROJECT_DIR con una ruta exacta."
        )

if _objetivo_instalacion is not None:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--upgrade",
            str(_objetivo_instalacion),
        ],
        check=True,
    )

for _modulo in tuple(sys.modules):
    if _modulo == "record_linkage" or _modulo.startswith("record_linkage."):
        del sys.modules[_modulo]
importlib.invalidate_caches()
record_linkage = importlib.import_module("record_linkage")

if record_linkage.__version__ != VERSION_OBJETIVO:
    raise RuntimeError(
        f"Se cargó rues-linker {record_linkage.__version__}; se requiere {VERSION_OBJETIVO}. "
        "Revise la ruta exacta y reinicie el kernel si actualizó una instalación cargada."
    )
ORIGEN_PAQUETE = Path(record_linkage.__file__).resolve()
print(f"rues-linker {record_linkage.__version__} | Python {sys.version.split()[0]}")
print(f"Origen verificado: {ORIGEN_PAQUETE}")

## 2. Configuración central - única celda que se edita

Los mapeos obligatorios fallan si no existen; los opcionales se conservan como payload cuando están presentes. Para añadir una fuente, agregue otro `SourceSpec` con un nombre único y ajuste `FUENTES_CONFIABLES`. Las variables de entorno permiten mover archivos sin editar rutas en el notebook.

In [ ]:
import hashlib
import shutil
import tempfile

from IPython.display import display

from record_linkage import ColumnType, Compression, IdentifierFormat, InputFormat, SourceSpec
from record_linkage.flujo import ConfigCruce, ResultadoCruceDisco, ejecutar_cruce
from record_linkage.ingestion import DuckDBIngestionSettings

# Entorno y rutas
MONTAR_DRIVE = False
COPIAR_SNAPSHOT_A_DRIVE = False
if MONTAR_DRIVE:
    if not ES_COLAB:
        raise RuntimeError("MONTAR_DRIVE solo aplica en Google Colab.")
    importlib.import_module("google.colab.drive").mount("/content/drive")

NOMBRE_CORRIDA = "rues_x_exportaciones"
_BASE_LOCAL = Path.cwd() / "data"
_BASE_PREDETERMINADA = (
    Path("/content/datos")
    if ES_COLAB
    else (_BASE_LOCAL if _BASE_LOCAL.is_dir() else Path.home() / "Downloads" / "rues-linker")
)
BASE_DATOS = (
    Path(os.environ.get("RUES_LINKER_DATA_DIR", str(_BASE_PREDETERMINADA))).expanduser().resolve()
)
RUTA_RUES = (
    Path(
        os.environ.get(
            "RUES_LINKER_RUES_PATH",
            str(BASE_DATOS / "2026-08-25 A00 - data_CSV_RUES.zip"),
        )
    )
    .expanduser()
    .resolve()
)
RUTA_EXPORTACIONES = (
    Path(
        os.environ.get(
            "RUES_LINKER_EXPORT_PATH",
            str(BASE_DATOS / "2026-08-21 Base_Exportaciones_Colombianas_2021-2026 (Junio).zip"),
        )
    )
    .expanduser()
    .resolve()
)
# La publicación generacional debe ocurrir en un filesystem local.
# Drive/FUSE se usa sólo para copiar un snapshot verificado al terminar.
_SALIDA_PREDETERMINADA = (
    Path("/content/resultados_rues_linker") if ES_COLAB else Path.cwd() / "resultados_rues_linker"
)
RUTA_SALIDA = (
    Path(os.environ.get("RUES_LINKER_OUTPUT_DIR", str(_SALIDA_PREDETERMINADA)))
    .expanduser()
    .resolve()
    / NOMBRE_CORRIDA
)
_SNAPSHOT_CONFIG = os.environ.get("RUES_LINKER_SNAPSHOT_DIR")
RUTA_SNAPSHOT_PERSISTENTE: Path | None = (
    Path(_SNAPSHOT_CONFIG).expanduser().resolve() / NOMBRE_CORRIDA
    if _SNAPSHOT_CONFIG
    else (
        Path("/content/drive/MyDrive/resultados_rues_linker_snapshots") / NOMBRE_CORRIDA
        if ES_COLAB
        else None
    )
)
RUTA_TRABAJO = (
    Path(
        os.environ.get(
            "RUES_LINKER_WORK_DIR",
            str(Path(tempfile.gettempdir()) / "rues_linker" / NOMBRE_CORRIDA),
        )
    )
    .expanduser()
    .resolve()
)

# Contratos: duplique un SourceSpec para agregar cualquiera de N fuentes.
FUENTES: list[SourceSpec] = [
    SourceSpec(
        name="RUES",
        path=RUTA_RUES,
        format=InputFormat.AUTO,
        compression=Compression.AUTO,
        encoding="auto",
        delimiter=",",
        column_mapping={
            "NIT": "NUMERO_IDENTIFICACION",
            "RAZON_SOCIAL": "RAZON_SOCIAL",
        },
        optional_column_mapping={
            "CIUDAD": "CODIGO_MUNICIPIO_COMERCIAL",
            "DIRECCION": "DIRECCION_COMERCIAL",
            "TELEFONO": "TELEFONO_COMERCIAL_1",
            "EMAIL": "CORREO_ELECTRONICO_COMERCIAL",
        },
        column_types={
            "NIT": ColumnType.IDENTIFIER,
            "RAZON_SOCIAL": ColumnType.STRING,
            "CIUDAD": ColumnType.STRING,
            "DIRECCION": ColumnType.STRING,
            "TELEFONO": ColumnType.STRING,
            "EMAIL": ColumnType.STRING,
        },
        identifier_formats={"NIT": IdentifierFormat(mode="digits", min_length=6, max_length=12)},
        null_values={
            "NIT": ("0", "00", "0000000000000"),
            "RAZON_SOCIAL": ("SIN DATO", "NO DEFINIDO"),
        },
    ),
    SourceSpec(
        name="EXPORTACIONES",
        path=RUTA_EXPORTACIONES,
        format=InputFormat.AUTO,
        compression=Compression.AUTO,
        encoding="auto",
        delimiter="\t",
        column_mapping={"NIT": "Nit Exportador", "RAZON_SOCIAL": "Razon Social"},
        optional_column_mapping={
            "DEPARTAMENTO": "Departamento Origen",
            "PAIS_DESTINO": "Pais Destino",
            "SECTOR": "Sector",
        },
        column_types={
            "NIT": ColumnType.IDENTIFIER,
            "RAZON_SOCIAL": ColumnType.STRING,
            "DEPARTAMENTO": ColumnType.STRING,
            "PAIS_DESTINO": ColumnType.STRING,
            "SECTOR": ColumnType.STRING,
        },
        identifier_formats={"NIT": IdentifierFormat(mode="digits", min_length=6, max_length=12)},
        null_values={
            "NIT": ("-1", "0", "00"),
            "RAZON_SOCIAL": ("NO DEFINIDO",),
        },
    ),
]

# Semántica del cruce
# Una fuente ejecuta deduplicación; dos o más ejecutan linkage automáticamente.
# La prioridad canónica sigue source_quality_weights del perfil; sin esos pesos, usa FUENTES.
FUENTES_CONFIABLES = {"RUES"}
VARIABLES_EXTRA: list[dict[str, object]] = []
PERFIL = "produccion_estandar"
AJUSTES_PERFIL = {
    "veto_nit_base_distinto": True,
    "cannot_link_identificador": True,
    "liberar_fuentes_tras_l1": True,
    "lsh_chunk_size": 75_000,
    "scoring_batch_size": 25_000,
}
PERFIL_MULTICAMPO = None

# Universo y controles de seguridad
CORRIDA_COMPLETA = False
LIMITE_MUESTRA_DEFAULT = 100_000
LIMITES_ESPECIFICOS = {"EXPORTACIONES": 200_000}
LIMITES_FILAS = (
    None
    if CORRIDA_COMPLETA
    else {spec.name: LIMITES_ESPECIFICOS.get(spec.name, LIMITE_MUESTRA_DEFAULT) for spec in FUENTES}
)
FILAS_SMOKE = 5_000
REUSAR_CHECKPOINTS = True
PRESERVAR_PAYLOAD = True
FILAS_PREVIEW = 25
FILAS_QA = 25
ESPACIO_MINIMO_TRABAJO_GIB = 2

DUCKDB_SETTINGS = DuckDBIngestionSettings(
    memory_limit="512MB",
    threads=2,
    temp_directory=RUTA_TRABAJO / "duckdb_spill",
    max_temp_directory_size="10GB",
    preserve_insertion_order=True,
    parquet_compression="zstd",
)

## 3. Preflight

Estas comprobaciones son baratas: validan cardinalidad, nombres, rutas, filesystem local y espacio antes de iniciar. Una fuente ejecuta deduplicación y dos o más ejecutan linkage. Construir `ConfigCruce` comprueba además que los parámetros están conectados al contrato real; `ejecutar_cruce` repetirá el preflight interno de la librería. La publicación ocurre localmente y, de forma opcional, se copia después como snapshot verificado a Drive.

In [ ]:
_nombres = [spec.name for spec in FUENTES]
if len(set(_nombres)) != len(_nombres):
    raise ValueError(f"Nombres de fuente duplicados: {_nombres}")
if not set(_nombres) >= FUENTES_CONFIABLES:
    raise ValueError("FUENTES_CONFIABLES contiene nombres no declarados.")
if LIMITES_FILAS is not None and set(LIMITES_FILAS) - set(_nombres):
    raise ValueError("LIMITES_FILAS contiene nombres no declarados.")
if Path(NOMBRE_CORRIDA).name != NOMBRE_CORRIDA:
    raise ValueError("NOMBRE_CORRIDA no puede contener componentes de ruta.")

_faltantes = [spec.path for spec in FUENTES if not spec.path.is_file()]
if _faltantes:
    detalle = "\n".join(f"  - {ruta}" for ruta in _faltantes)
    raise FileNotFoundError(f"Faltan fuentes configuradas:\n{detalle}")
if RUTA_SALIDA == RUTA_TRABAJO:
    raise ValueError("RUTA_SALIDA y RUTA_TRABAJO deben ser distintas.")
_drive_colab = Path("/content/drive").resolve()
if ES_COLAB and RUTA_TRABAJO.is_relative_to(_drive_colab):
    raise ValueError("RUTA_TRABAJO debe ser disco local, no Drive/FUSE.")
if ES_COLAB and RUTA_SALIDA.is_relative_to(_drive_colab):
    raise ValueError(
        "RUTA_SALIDA debe ser local para que lock/replace/fsync sean fiables; "
        "use COPIAR_SNAPSHOT_A_DRIVE después del commit."
    )
if COPIAR_SNAPSHOT_A_DRIVE:
    if not ES_COLAB or not Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("Monte Drive antes de activar COPIAR_SNAPSHOT_A_DRIVE.")
    if RUTA_SNAPSHOT_PERSISTENTE is None or not RUTA_SNAPSHOT_PERSISTENTE.is_relative_to(
        _drive_colab
    ):
        raise ValueError("RUTA_SNAPSHOT_PERSISTENTE debe estar dentro de Drive.")

RUTA_SALIDA.mkdir(parents=True, exist_ok=True)
RUTA_TRABAJO.mkdir(parents=True, exist_ok=True)
Path(DUCKDB_SETTINGS.temp_directory).mkdir(parents=True, exist_ok=True)
_libre_gib = shutil.disk_usage(RUTA_TRABAJO).free / 1024**3
if _libre_gib < ESPACIO_MINIMO_TRABAJO_GIB:
    raise OSError(
        f"Solo hay {_libre_gib:.1f} GiB libres en {RUTA_TRABAJO}; "
        f"se requieren al menos {ESPACIO_MINIMO_TRABAJO_GIB} GiB."
    )

CFG = ConfigCruce(
    fuentes=FUENTES,
    workspace=RUTA_SALIDA,
    confiables=FUENTES_CONFIABLES,
    dir_trabajo=RUTA_TRABAJO,
    perfil=PERFIL,
    col_nit="NIT",
    col_nombre="RAZON_SOCIAL",
    col_ciudad=None,
    variables_extra=VARIABLES_EXTRA or None,
    filas_smoke=FILAS_SMOKE,
    exportar_excel=False,
    reusar_checkpoints=REUSAR_CHECKPOINTS,
    colapsar_duplicados_exactos=True,
    limite_filas=LIMITES_FILAS,
    ajustes_perfil=AJUSTES_PERFIL,
    perfil_multicampo=PERFIL_MULTICAMPO,
    motor_ingesta="duckdb",
    duckdb_settings=DUCKDB_SETTINGS,
    modo_resultado="disco",
    preservar_payload=PRESERVAR_PAYLOAD,
)

print(f"Fuentes ({len(FUENTES)}):")
for _spec in FUENTES:
    print(f"  {_spec.name}: {_spec.path} ({_spec.path.stat().st_size / 1024**2:,.1f} MiB)")
print(f"Salida: {RUTA_SALIDA}")
print(f"Trabajo local: {RUTA_TRABAJO} ({_libre_gib:.1f} GiB libres)")
print(f"DuckDB: RAM={DUCKDB_SETTINGS.memory_limit}, hilos={DUCKDB_SETTINGS.threads}")
print(f"Límites de filas: {LIMITES_FILAS or 'universo completo'}")
print(f"Payload preservado: {PRESERVAR_PAYLOAD}")
print(
    f"Snapshot persistente: {RUTA_SNAPSHOT_PERSISTENTE if COPIAR_SNAPSHOT_A_DRIVE else 'desactivado'}"
)

## 4. Ejecución

La salida debe ser `ResultadoCruceDisco`; ningún DataFrame final se conserva en el notebook.

In [ ]:
resultado = ejecutar_cruce(CFG)
if not isinstance(resultado, ResultadoCruceDisco):
    raise TypeError(f"Se esperaba ResultadoCruceDisco, no {type(resultado).__name__}.")
print(resultado.resumen())
print(resultado.tiempos())

## 5. Preview acotado

`TablaParquet.preview()` ejecuta `LIMIT` en DuckDB. No use `to_pandas()` con resultados productivos.

In [ ]:
for _nombre_tabla, _tabla in (
    ("golden", resultado.golden),
    ("correlativa", resultado.correlativa),
):
    if not _tabla.path.is_file():
        raise FileNotFoundError(_tabla.path)
    print(f"{_nombre_tabla}: {_tabla.rows:,} filas | {_tabla.path}")

print("Preview golden:")
display(resultado.golden.preview(rows=FILAS_PREVIEW))
print("Preview correlativa:")
display(resultado.correlativa.preview(rows=FILAS_PREVIEW))


## 6. QA N-fuente y artefactos

Los conteos y cruces se calculan sobre Parquet. La matriz crece con el número de fuentes, no con el número de registros; las entidades ausentes se limitan explícitamente por fuente.

In [ ]:
if resultado.correlativa.rows != resultado.metricas["filas_entrada"]:
    raise RuntimeError("La correlativa no conserva una fila por registro de entrada.")
if resultado.golden.rows != resultado.metricas["entidades"]:
    raise RuntimeError("El golden no conserva una fila por entidad.")

# El entregable: qué identidad quedó para cada grupo. `identidad_adoptada`
# FALLA si falta alguna de las cuatro columnas del contrato (ADR-0008),
# así que verificarlo y mostrarlo son la misma línea.
print("Identidad adoptada por grupo:")
display(resultado.identidad_adoptada(limite=FILAS_QA))

_cruce_fuente = resultado.cruce_por_fuente()
if set(_cruce_fuente["FUENTE"]) != set(_nombres):
    raise RuntimeError("El QA por fuente no contiene exactamente las fuentes declaradas.")
print("Cruce por fuente:")
display(_cruce_fuente)
print("Matriz de presencia N-fuente:")
display(resultado.matriz_presencia())

for _spec in FUENTES:
    print(f"Entidades ausentes de {_spec.name} (máximo {FILAS_QA}):")
    display(resultado.entidades_ausentes_de(_spec.name, limit=FILAS_QA))


def _sha256_archivo(_ruta: Path) -> str:
    _digest = hashlib.sha256()
    with _ruta.open("rb") as _stream:
        while _bloque := _stream.read(8 * 1024**2):
            _digest.update(_bloque)
    return _digest.hexdigest()


print("Artefactos reproducibles:")
_rutas_publicadas: dict[str, Path] = {}
_publicables = {k: v for k, v in resultado.rutas.items() if k != "dir_trabajo"}  # L1…L5 no se publica
for _nombre_ruta, _ruta_original in sorted(_publicables.items()):
    _ruta = Path(_ruta_original)
    if _nombre_ruta == "generation_dir":
        if not _ruta.is_dir():
            raise FileNotFoundError(_ruta)
    elif not _ruta.is_file():
        raise FileNotFoundError(_ruta)
    _rutas_publicadas[_nombre_ruta] = _ruta
    print(f"  {_nombre_ruta}: {_ruta}")

if COPIAR_SNAPSHOT_A_DRIVE:
    if RUTA_SNAPSHOT_PERSISTENTE is None:
        raise RuntimeError("Falta RUTA_SNAPSHOT_PERSISTENTE.")
    _destino_snapshot = RUTA_SNAPSHOT_PERSISTENTE / Path(resultado.rutas["generation_dir"]).name
    if _destino_snapshot.exists():
        raise FileExistsError(f"El snapshot ya existe: {_destino_snapshot}")
    _destino_snapshot.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(RUTA_SALIDA, _destino_snapshot)
    for _nombre_ruta, _origen in _rutas_publicadas.items():
        _copia = _destino_snapshot / _origen.relative_to(RUTA_SALIDA)
        if _nombre_ruta == "generation_dir":
            if not _copia.is_dir():
                raise RuntimeError(f"Snapshot incompleto: {_copia}")
        elif not _copia.is_file() or _sha256_archivo(_copia) != _sha256_archivo(_origen):
            raise RuntimeError(f"Checksum de snapshot inválido: {_copia}")
    print(f"Snapshot persistente verificado: {_destino_snapshot}")
elif ES_COLAB:
    print("ATENCIÓN: la salida está en disco efímero; descárguela o active el snapshot a Drive.")

## Notas de operación

- `memory_limit` gobierna el buffer manager de DuckDB, no toda la memoria del proceso Python.
- `LIMITES_FILAS` se aplica en staging. Con límites activos, los resultados son de ensayo y no deben presentarse como universo completo.
- `PRESERVAR_PAYLOAD=True` conserva las columnas declaradas que no participan en el matcher y las re-adjunta por fila original.
- Trabajo, spill y publicación generacional deben permanecer en disco local. Lock, `replace` y `fsync` no tienen garantías equivalentes sobre Drive/FUSE.
- En Colab, active `COPIAR_SNAPSHOT_A_DRIVE` sólo para copiar después del commit una instantánea única; la celda compara SHA-256 de cada artefacto.
- El JSON de metadatos registra parámetros, fuentes, límites, métricas y tiempos de la corrida.